# Notebook 03 — Feature Engineering: Lags, Rolling Windows, and Calendar Signals

This notebook covers Objective 3 of the project synopsis: turning the cleaned, store-merged
Rossmann frame produced in notebook 02 (`train_merged_clean.parquet`) into the model-ready
feature matrix the rest of the project trains and evaluates on. Three feature groups get added on
top of the cleaned columns:

1. **Calendar indicators** — `Year`, `Month`, `Day`, `WeekOfYear`, `Quarter`, `is_weekend`,
   `is_month_start`/`is_month_end`, and `is_day_before_holiday`/`is_day_after_holiday`.
2. **Lag variables** — `lag_1`, `lag_7`, `lag_14`, `lag_28`: the store's own Sales value exactly
   1 / 7 / 14 / 28 days earlier, capturing short-term persistence and weekly seasonality.
3. **Rolling-window statistics** — `roll_mean_7/14/28`, `roll_std_7/14/28`, `roll_min_7`,
   `roll_max_7`, and `promo_last_7_days_count`: smoothed summaries of recent sales level,
   volatility, and promo exposure.

The target itself is also transformed. Notebook 02's EDA showed `Sales` is strongly right-skewed —
a long tail of high-volume store-days — which would let a handful of large stores dominate any
loss function trained on the raw scale. Here we add `log_sales = log1p(Sales)` so modeling and
evaluation downstream (notebooks 04-08) work against a variance-stabilized target, with
`np.expm1()` used to invert back to the Sales scale wherever metrics or plots need the original
units.

Because the lag/rolling features are only safe to use if they never peek at a store's own future
Sales, this notebook also runs the executable leakage-prevention check that every later notebook
implicitly relies on.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

The feature-computation logic itself is not re-implemented here — it already lives in
`src/features.py` (and the parquet I/O in `src/data_io.py`), both already unit-tested in
`tests/test_features.py`. This notebook's job is to call that tested logic, inspect its output,
and independently verify the leakage guarantees it claims to provide.

In [ ]:
from src.data_io import load_parquet, save_parquet
from src.features import build_feature_matrix, filter_trainable_rows, assert_no_leakage, REQUIRED_FEATURE_COLS

Load the cleaned, merged frame notebook 02 wrote to `data/interim/train_merged_clean.parquet`.
That frame already has the store attributes merged in, `StateHoliday` encoded into
`state_holiday_type`/`is_state_holiday`, and the competition/Promo2 fields imputed and derived —
and, importantly, it still has every `Open == 0` (closed-store) row intact. Keeping closed days in
each store's date index is what lets the lag/rolling windows below walk back in true calendar-day
order without silently skipping over closures.

In [ ]:
train_clean = load_parquet("../data/interim/train_merged_clean.parquet")

`build_feature_matrix` sorts by `(Store, Date)`, then layers on the calendar columns, the four lag
columns, and the nine rolling-window columns, and finally derives `log_sales` and a boolean
`is_trainable` flag. It deliberately returns the **full** frame — closed-store rows and each
store's incomplete early-history rows included — rather than a pre-filtered one, because both the
leakage check and the missing-value diagnostic below need that complete per-store history to check
against.

Printing the per-column NaN share over `REQUIRED_FEATURE_COLS` is the expected sanity check here:
`lag_1` should have the smallest share (only the single earliest day per store is missing it), and
`lag_28`/`roll_mean_28`/`roll_std_28` the largest, since a full 28-day warm-up window is the
hardest history requirement to satisfy at the start of each store's record.

In [ ]:
full = build_feature_matrix(train_clean)
print(f"Engineered frame shape: {full.shape}")
full[REQUIRED_FEATURE_COLS].isna().mean().sort_values(ascending=False)

## Leakage prevention

Two different lookahead rules are in play here, and they are not the same thing.

1. **The Sales-derived features must never look forward.** `lag_k` and every `roll_*` column are
   computed by first shifting the relevant series (`Sales`, or `Promo` for the promo-count
   feature) by 1 day **before** any windowing happens (see `add_lag_features` /
   `add_rolling_features` in `src/features.py`). That shift-then-window order means the value
   stored for `(store, date)` is built exclusively from that same store's rows with a strictly
   earlier date. If the shift were applied after the rolling window instead of before it, today's
   own Sales would leak into today's `roll_mean_7`, and an offline backtest would look far better
   than a real forecast ever could.

2. **The calendar/holiday features are allowed to look 1 day ahead, and that is not leakage.**
   `is_day_before_holiday` and `is_day_after_holiday` are built with `shift(-1)` / `shift(1)` on
   the holiday flag. That is safe specifically because public/school holiday calendars are
   *exogenous* and known long in advance — on any real forecast date we already know whether
   tomorrow is a holiday, the same way we already know what day of the week it is. That is
   fundamentally different from a feature that would need to know tomorrow's *Sales*, which is
   exactly the quantity we are trying to predict and could never know in advance.

`assert_no_leakage` turns rule (1) into an executable proof rather than just a design claim: for a
random sample of `(store, date)` rows it discards the stored `lag_7`/`roll_mean_7` values,
re-walks that store's raw Sales history strictly before `date`, recomputes `lag_7` and
`roll_mean_7` from scratch off that raw history, and asserts the two numbers match to
floating-point tolerance. If the shift-before-window order in `src/features.py` were ever broken
by a future edit, this call would fail immediately rather than silently inflating every downstream
model's offline accuracy.

In [ ]:
assert_no_leakage(full, sample_size=30)
print("Leakage check passed: lag_7/roll_mean_7 independently recomputed from raw per-store "
      "history match the stored feature columns for 30 random (store, date) samples.")

With the leakage check passed, the next question is how much of the engineered frame is actually
usable for training. `is_trainable` is `True` only when a row is both an open-store day
(`Open == 1` — a closed store has no demand signal for a Sales model to learn from) *and* has
every one of the `REQUIRED_FEATURE_COLS` populated (no missing lag/rolling values left over from
the warm-up period checked above).

In [ ]:
trainable_share = full["is_trainable"].mean()
print(f"{trainable_share:.1%} of rows are trainable (Open==1 and all lag/rolling features present)")

Two distinct groups of rows fail `is_trainable`, for two distinct reasons, and it's worth not
conflating them. The larger group is simply every closed-store row (`Open == 0`); excluding those
isn't really a feature-engineering concern at all — it's the definition of the task, since a
closed store sold nothing, so there is no demand pattern there for a Sales model to learn from.

The smaller group is the one this notebook actually introduces: roughly 3% of all rows are
open-store days that still get excluded, purely because they fall inside the first 28 days of
their store's history and therefore cannot yet have a complete `lag_28`/`roll_mean_28` (or, closer
to day 1, a complete `lag_7`/`roll_mean_7`/`lag_14`/`roll_mean_14` either). There simply isn't 28
prior calendar days of that store's Sales to draw on yet. With a 28-day warm-up window and this
sample's fifteen stores, that is on the order of 28 x 15 ≈ 420 row-days before accounting for any
overlap with days already excluded as closed — which lines up with the small residual gap between
the `is_trainable` share above and the much larger, Open==0-driven share of excluded rows.

Dropping these warm-up rows outright, rather than imputing the missing lag/rolling values, is the
right call. Any imputed value — backfilling with the store's global mean, zero-filling, or
anything else — would not represent an actual recent-history observation, so a model trained on it
would be learning from a fabricated signal in exactly the columns that are supposed to encode
genuine recent demand. A few hundred unusable row-days is a negligible cost against that risk: it
is better for the model to never see a `(store, date)` pair than to see one built on invented
history.

In [ ]:
trainable = filter_trainable_rows(full)
print(f"Trainable rows: {len(trainable)} (dropped {len(full) - len(trainable)})")

save_parquet(trainable, "../data/processed/train_features.parquet")

## Output

`train_features.parquet` now holds the trainable subset of the engineered frame: every row has
`Open == 1` and a complete set of calendar/lag/rolling features, with `log_sales` ready to use as
the regression target, and with the leakage guarantees above independently verified rather than
assumed. This is the exact file notebook 04's baselines (`seasonal_naive_predict`,
`moving_average_predict`) and the later Ridge/LightGBM notebooks load from
`../data/processed/train_features.parquet`, training on the shared `NUMERIC_FEATURE_COLS` /
`CATEGORICAL_FEATURE_COLS` / `ALL_FEATURE_COLS` contracts defined alongside these features in
`src/features.py`, so every model downstream sees an identical, leakage-checked column set.